# Why was pydantic needed?

In [1]:
def insert_patient(name, age):
    print(name)
    print(age)

    print('New patient inserted')

patient1 = insert_patient(name  = 'Nikunj', age = 'Twenty four')

# See we expected 'age' to be an integer, but the user inserted string. Even then it worked.
# So, to solve this, we can impliment type hinting

Nikunj
Twenty four
New patient inserted


In [2]:
def insert_patient(name: str, age: int):
    print(name)
    print(age)

    print('New patient inserted')

insert_patient('Nikunj', 'Twenty-four')

# See, now this functions suggests the data type of both the parameters but still, there's no restriction.
# User can insert any data type in these parameters

Nikunj
Twenty-four
New patient inserted


# So, to overcome this limitation, we can use pydantic which does `type-checking` before moving forward.

# We need to inherit class named `BaseModel` from pydantic package.

In [3]:
from pydantic import BaseModel

In [4]:
class Patient(BaseModel):

    name: str
    age: int

# Now, this function will not accept 'name' and 'age' as its parameters.
# Instead, it will take pydantic's object as its parameter
def insert_patient(patient: Patient):
    print(patient.name)
    print(patient.age)

    print('New patient inserted')


In [5]:
# p1 is a pydantic's object
p1 = Patient(**{'name': 'Nikunj', 'age': 30})

# Passing p1(a pydantic's object) as an argument to insert_patient

insert_patient(p1)

Nikunj
30
New patient inserted


In [ ]:
# Now, if we insert any other dtype in age or name, it will through an error.
# Also, it automatically converts str '30' to int 30

p = {'name': 'Nikunj', 'age': 'Twenty-four'}

p1= Patient(**p)

# See, we're getting validation error for 'age'

ValidationError: 1 validation error for Patient
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Twenty-four', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [ ]:
# Now, if we insert and other dtype in age or name, it will throw an error.
# Also, it automatically converts str '30' to int 30

p = {'name': 'Nikunj', 'age': '24'}

p1= Patient(**p)

print(type(p1.age))    # See, 24 got converted to int from str
print()

insert_patient(p1)


<class 'int'>

Nikunj
24
New patient inserted


# This was just an introduction to pydantic. 
# Now, let's learn it in detail.

In [8]:
from pydantic import BaseModel
from typing import Dict, List, Optional

In [9]:
class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool = False  
    contact_details: Dict[str, str]
    education: Optional[List[str]] = None    # Whenever we use Optional, it is compulsoory to pass a default Value

# Why are not we using simple list but List? We will see this in the end.

def insert_patient(x: Patient):
    print(x.name)
    print(x.age)
    print(x.weight)
    print(x.married)
    print(x.contact_details)
    print(x.education)

# By-defauld, everything is required in pydantic, To make it optional, use Optional class

In [14]:
patient_info = {
    'name': 'Harshit',
    'age': 24,
    'weight': 65,
    'married': False,
    'contact_details': {'email': 'abc@gmail.com', 'mob_no': '7383196410'},
    'education': ['MBBS', '12th Science', '10th']
}
patient_info2 = {
    'name': 'Harshit',
    'age': 24,
    'weight': 65,
    #'married': False,    By-default, False
    'contact_details': {'email': 'abc@gmail.com', 'mob_no': '7383196410'},
    #'education': ['MBBS', '12th Science', '10th']
}
    
    
    

p1 = Patient(**patient_info)
p2 = Patient(**patient_info2)

In [15]:
insert_patient(p1)

print()
print()

insert_patient(p2)

Harshit
24
65.0
False
{'email': 'abc@gmail.com', 'mob_no': '7383196410'}
['MBBS', '12th Science', '10th']


Harshit
24
65.0
False
{'email': 'abc@gmail.com', 'mob_no': '7383196410'}
None


# Now, our question, why are not we using simple list and dict but List and DICT?

* See, here we are ensuring that for `education` the dtype should be list and it should be a list of strings <br>
* Similarly, for `contact_details`, it should be dictionary with keys and values as string.

With simple list and dict, we can ensure that the dtype should be list and dictionary, but we cannot ensure the dtypes of values stored inside these list and dict.

# Pydantic datatypes pydantic has its own data types. They're
1) EmailStr
2) AnyUrl
3) Field

In [70]:
from pydantic import BaseModel, EmailStr, AnyUrl
from typing import List, Dict, Optional

In [73]:
class Student(BaseModel):
    roll_no: int
    name: str
    surname: str
    linkedin: Optional[AnyUrl] = None
    standard: str
    email_id: EmailStr
    subjects: Optional[List[str]] = 'Not Available'

def insert_student(stud: Student):

    print(stud.roll_no)
    print(stud.name)
    print(stud.surname)
    print(stud.linkedin)
    print(stud.email_id)
    print(stud.standard)
    print(stud.subjects)    

In [74]:
stud_info1 = {
    'name': 'Nikunj',
    'surname': 'Bosamiya',
    'roll_no': 26,
    'standard': '10th',
    'email_id': 'nikss.2611@gmail.com',
    'linkedin': 'http://linkedin.com/352'
}

stud1 = Student(**stud_info1)

insert_student(stud1)

26
Nikunj
Bosamiya
http://linkedin.com/352
nikss.2611@gmail.com
10th
Not Available


# Suppose, we wanna keep constraint that `Age` should not be less than 0 and greater than 30, we will gave to use `Field` class

In [4]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

In [5]:
ann_str = Annotated[str, Field(description = '')]
ann_rno = Annotated[int, Field(description = 'Roll number of the student')]

class Student(BaseModel):
    roll_no: ann_rno
    name: Annotated[str, Field(description = 'Name of a student', max_length = 10)]
    surname: ann_str
    age: Annotated[float, Field(gt = 0, lt = 30)]
    linkedin: Annotated[Optional[AnyUrl],  Field(description = 'Linkedinn-id of the student', title = 'Student name', default = None)]
    standard: ann_str    # This is the benefit of Annotated, we can reuse validation_checks and constraints.
    email_id: Annotated[EmailStr, Field(description = 'Email-id of the student')]
    subjects: Annotated[Optional[List[str]], Field(description = 'subjects of the student', title = 'Name of subjects',
                                                   example = ['Python', 'Statistics'], default = None)]

def insert_student(s: Student):

    for key, value in s.model_dump().items():    # model_dump() converts pydantic's object to dictionary (same as dict())
        print(f'{key}: {value}')

    print()
    print()
    print()
    print('Student inserted successfully')

In [6]:
stud_info1 = {
    'name': 'Nikunj',
    'surname': 'Bosamiya',
    'roll_no': 26,
    'age': 24,
    'standard': '10th',
    'email_id': 'nikss.2611@gmail.com',
    'linkedin': 'http://linkedin.com/352'
}

stud1 = Student(**stud_info1)

insert_student(stud1)

roll_no: 26
name: Nikunj
surname: Bosamiya
age: 24.0
linkedin: http://linkedin.com/352
standard: 10th
email_id: nikss.2611@gmail.com
subjects: None



Student inserted successfully


In [4]:
# This is also available in Pydantic's object

stud1.model_dump()

{'roll_no': 26,
 'name': 'Nikunj',
 'surname': 'Bosamiya',
 'age': 24.0,
 'linkedin': AnyUrl('http://linkedin.com/352'),
 'standard': '10th',
 'email_id': 'nikss.2611@gmail.com',
 'subjects': None}

# Why to use Annotated?

1) Annotated combines type and metadata in single frame. Without annotated, type will be on left and metedata on right

`Without Annotated` <br>
name: str = Field(description = 'Name of the student', max_length = 20) <br>

`With Annotated`<br>
name: Annotated[str, Field(description = 'Name of the student', max_length = 20)]

2) We can reuse. <br>
ann = Annotated[List[str], Field(description = 'List of subjects', max_length = 5)]

subjects = ann

3. In FastAPI and other tools, if not used Annotated, it creates much issues.

`So, it is always advised to used Annotated`

# Paremeters of Field()

* `description`: Add description of the parameter
* `max_length`: Maximum length allowed
* `min_length`: Minimum length allowed
* `lt`: Values should be less_than
* `gt`: Values should be greater_than
* `le/ge`: Values should be greater than/less than equal_too.
* `example`: Can add example for expected values for the user's convenience
* `default`: Set default value
* `title`: Give title to the Parameter
* `strict (True/ False)`: Used to implement strict type validation. If 'int', '12' will not be allowed which otherwise will.

# `Part 2`: Field Validator

Supose, we are running a college, and college only accepts addmisions of students who scored 80% or more in 12th.

Now, the application is done online, how will be differentiate the students based on their 12th percentage, like, if percentage is less than 80, then 'Not eligible' else 'Eligible'?

`Here comes **field_validator** into play.`

Field(le, gt) is used for common use case, but if we wanna apply custom, business based data validation. we need field_validator. <br>

For example:
1. IF percentage is >80, Eligible for admission else not eligible.
2. Student's name should be converted to captial case.
3. etc

In [5]:
from pydantic import field_validator

In [ ]:
class Student(BaseModel):

    name: Annotated[str, Field(description = 'Name of the student')]
    surname: Annotated[str, Field(description = 'Surname of the student')]

    age: Annotated[int, Field(description = 'Age of the student')]

    percentage: Annotated[float, Field(description = '12th Percentage')]

    # To add custom validator, we create a method inside the class and it should be used with decorator field_validator

    @field_validator('percentage')
    @classmethod
    def age_validator(cls, value):

        if value >=80:
            print('You are eligible for the admission')
        else:
            print('You are not eligible for the admission')

        return value

    @field_validator('name')
    @classmethod
    def trans_name(cls, value):
        return value.upper()
        

In [11]:
def check_elig(s: Student):
    for key, value in s.model_dump().items():
        print(f'{key}: {value}')

    print()
    print()
    print()

    print('Student successfully added.')

In [12]:
data = {
    'name': 'Nikunj',
    'surname': 'Bosamiya',
    'age': 24,
    'percentage': 80
}

s1 = Student(**data)

You are eligible for the admission


In [18]:
print(s1)

name='NIKUNJ' surname='Bosamiya' age=24 percentage=80.0


# Model validator:

In our previous code, we used field_validator to validate a single parameter. but what if we wanna validate based on two fields?

For example: if percentage is <80 father's income should be > 600000 PA.

`To do this, we use model_validator. (See the below code)`

In [19]:
from pydantic import model_validator

In [3]:
class Student(BaseModel):
    name: Annotated[str, Field(description = 'Name of the student')]
    surname: Annotated[str, Field(description = 'Surname of the student')]

    percentage: Annotated['float', Field(description = "Percentage of the student")]

    income: Annotated[float, Field(description = "Father's Income", example = '2.5, 6.5. 8.5')]


    @model_validator(mode = 'after')
    @classmethod
    def validate_admission(cls, model):

        if model.percentage >= 80:
            print('You are eligible for the admission')
        elif model.percentage < 80 and model.income >= 6:
            print('You are eligible for the admission')
        else:
            print('You are not eligible for the admission')
        
        return model


In [6]:
data = {
    'name': 'Nikunj',
    'surname': 'Bosamiya',
    'percentage': 85,
    'income': 2
}

s1 = Student(**data)

You are eligible for the admission


In [39]:
s1

Student(name='Nikunj', surname='Bosamiya', percentage=85.0, income=2.0)

# Next:

1) Compute Fields
2) Nested Models
3) Serialization

# Computed Fields:
This is such field in pydantic, whose value is not provided by the user. We create it from the other fields. <br>
`For example`: length * breath = carpet_area

In [7]:
from pydantic import computed_field

In [8]:
class Student(BaseModel):
    roll_no: Annotated[int, Field(gt = 0)]
    name: Annotated[str, Field(description = 'Name of the student', max_length = 20)]

    total_marks: Annotated[int, Field(gt = 0)]
    total_subjects: Annotated[int, Field(gt = 0)]

    @computed_field
    @property
    def calc_avg(self) -> float:
        average = (self.total_marks / self.total_subjects)

        return round(average, 2)

    # The 'key' of custom field will be the name of the method (Here, calc_percentage)

In [9]:
def add_student(s: Student):

    for key, value in s.model_dump().items():
        print(f'{key}: {value}')

    print()
    print()
    print()

    print('Student successfully added.')

In [10]:
data = {
    'name': 'Nikunj',
    'roll_no': 26,
    'total_marks': 80,
    'total_subjects': 6
}

add_student(Student(**data))

roll_no: 26
name: Nikunj
total_marks: 80
total_subjects: 6
calc_avg: 13.33



Student successfully added.


# Nested Model

In [49]:
class Address(BaseModel):

    city: str
    state: str
    pincode: int

class Student(BaseModel):

    name: Annotated[str, Field(description = 'Name of the student')]
    surname: Annotated[str, Field(description = 'Surname of the student')]
    gender: str = 'Male'

    age: Annotated[int, Field(description = 'Age of the student')]
    address: Address

In [50]:
add1 = Address(city = 'Jetpur', state = 'Gujarat', pincode = 360370)

add1

Address(city='Jetpur', state='Gujarat', pincode=360370)

In [51]:
s1 = Student(
    name = 'Nikunj',
    surname = 'Bosamiya',
    age = 24,
    address = add1
)



In [ ]:
print(s1)
print()

print(s1.name)
print()

print(s1.address)
print()

print(s1.address.pincode)
print()

print(s1.model_dump(include = ['name', 'age']))
print()

print(s1.model_dump(exclude = ['name']))
print()

print(s1.model_dump(exclude = {'address': 'pincode'}))
print()

print(s1.model_dump(exclude_unset = True))
print()
print()

'''exclude_unset = True

If we have given any default value to a parameter and if we dont implicitly give any value to this parameter,
it will be set to default value. Right?

exclude_unset will not return such parameters which is not passed while creating the object. 

'''

name='Nikunj' surname='Bosamiya' gender='Male' age=24 address=Address(city='Jetpur', state='Gujarat', pincode=360370)

Nikunj

city='Jetpur' state='Gujarat' pincode=360370

360370

{'name': 'Nikunj', 'age': 24}

{'surname': 'Bosamiya', 'gender': 'Male', 'age': 24, 'address': {'city': 'Jetpur', 'state': 'Gujarat', 'pincode': 360370}}

{'name': 'Nikunj', 'surname': 'Bosamiya', 'gender': 'Male', 'age': 24, 'address': {'city': 'Jetpur', 'state': 'Gujarat'}}

{'name': 'Nikunj', 'surname': 'Bosamiya', 'age': 24, 'address': {'city': 'Jetpur', 'state': 'Gujarat', 'pincode': 360370}}




'exclude_unset = True\n\nIf we have given any default value to a parameter and if we dont implicitly give any value to this parameter,\nit will be set to default value. Right?\n\nexclude_unset will not return such  parameters which is not passed while creating the object. \n\n'

In [2]:
# Which classes are under which package

from pydantic import BaseModel, EmailStr, AnyUrl, Field, computed_field, field_validator, model_validator

from typing import Dict, List, Optional, Annotated

# Session Complete